In [1]:
# app.py
import io
import re
from datetime import datetime

import pandas as pd
import streamlit as st
from PyPDF2 import PdfReader


# ---------------------------
# Streamlit Page Config
# ---------------------------
st.set_page_config(page_title="BOL → Inventory Sheet", page_icon="📦", layout="centered")
st.title("📦 BOL → Inventory Sheet")
st.write(
    "Drag & drop your **BOL PDFs** below. The app will parse them and generate the Excel file "
    "in the format we finalized (Received By in column A + Notes at the end)."
)


# ---------------------------
# Utilities
# ---------------------------
def extract_pdf_text_from_bytes(file_bytes: bytes) -> str:
    """Extract plain text from a PDF (bytes) using PyPDF2."""
    reader = PdfReader(io.BytesIO(file_bytes))
    chunks = []
    for page in reader.pages:
        chunks.append(page.extract_text() or "")
    return "\n".join(chunks)


def _find(pattern: str, text: str, default: str = "") -> str:
    """Convenience regex finder that returns first capture group or default."""
    m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else default


def parse_bol_text(pdf_text: str) -> list[dict]:
    """
    Parse one BOL's text into a row.
    Tailored to the BOLs you provided:
    - Uses 'B/L NO.' and 'B/L DATE'
    - Pulls Product, Quantity, Packaging (from the description block)
    - Pulls Lot No (if present)
    - Pulls Ship To City/State from the top-left Ship-To block above 'USA'
    """
    rows = []

    # Split blocks at each B/L NO. — stable anchor across pages
    parts = re.split(r"(?=B/L NO\.)", pdf_text, flags=re.IGNORECASE)
    for block in parts:
        if "B/L NO." not in block:
            continue

        # -------- B/L No & Date --------
        bl_no = _find(r"B/L NO\.\s*([0-9]+)", block)
        bl_date_raw = _find(r"B/L DATE\s*([0-9/\-]{8,10})", block)
        bl_date = bl_date_raw
        for fmt in ("%m/%d/%Y", "%m-%d-%Y", "%Y-%m-%d"):
            try:
                bl_date = datetime.strptime(bl_date_raw, fmt).strftime("%m/%d/%Y")
                break
            except Exception:
                pass  # try next

        # -------- Product --------
        # Usually appears after the "Product" label and before Transporter
        product = _find(r"Product\s*(.*?)\s*Transporter", block)

        # -------- Quantity --------
        # Look around the order/quantity table area
        quantity = _find(r"\bQUANTITY\s*[\s\S]{0,40}?([0-9]{1,3})(?![0-9])", block)

        # -------- Packaging + Description --------
        # Try to capture a single concise packaging descriptor (DRUM/TOTE/PAIL + weight) when present.
        packaging = _find(
            r"PACKAGING\s*[\s\S]*?DESCRIPTION\s*(.*?)\s*(?:NET|Total:)",
            block
        )
        if not packaging:
            # Fallback if 'PACKAGING' header didn't OCR nicely
            packaging = _find(r"DESCRIPTION\s*(.*?)(?:NET|Total:)", block)
        packaging = (packaging or "").replace("\n", " ").strip()

        # Clean overly long packaging captures (optional heuristic)
        packaging = re.sub(r"\s{2,}", " ", packaging)

        # -------- Lot No --------
        lot_no = _find(r"Lot No\.\s*([A-Za-z0-9\-; ]+)", block)

        # -------- Ship-To City/State (top-left block above 'USA') --------
        # Normalize whitespace to simplify matching lines that wrapped
        cleaned = " ".join(block.split())
        ship_to_body = _find(r"Barton Solvents, Inc\.\s*(.*?)\s*USA", cleaned)
        city_state = ""
        if ship_to_body:
            m = re.search(r"([A-Za-z][A-Za-z ]+),\s*([A-Z]{2})\b", ship_to_body)
            if m:
                city_state = f"{m.group(1).strip()}, {m.group(2).strip()}"

        rows.append({
            "B/L No": bl_no,
            "B/L Date": bl_date,
            "Product": product,
            "Packaging": packaging,
            "Quantity": quantity,
            "Lot No": lot_no,
            "Ship To City/State": city_state
        })

    return rows


def process_bol_files(files: list[io.BytesIO]) -> pd.DataFrame:
    """Takes uploaded PDF files (BytesIO) → returns the final dataframe."""
    all_rows = []
    for f in files:
        text = extract_pdf_text_from_bytes(f.getvalue())
        rows = parse_bol_text(text)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    # Ensure B/L Date is MM/DD/YYYY
    if "B/L Date" in df.columns:
        df["B/L Date"] = pd.to_datetime(df["B/L Date"], errors="coerce").dt.strftime("%m/%d/%Y")

    # Add 'Received By' as Column A
    df.insert(0, "Received By", "")

    # Add 'Notes' at end
    df["Notes"] = ""

    # Column order we agreed on
    preferred = ["Received By", "B/L No", "B/L Date", "Product", "Packaging",
                 "Quantity", "Lot No", "Ship To City/State", "Notes"]
    existing = [c for c in preferred if c in df.columns]
    rest = [c for c in df.columns if c not in existing]
    df = df[existing + rest]

    return df


def to_excel_bytes(df: pd.DataFrame) -> bytes:
    """Save dataframe to an in‑memory Excel file, with light formatting."""
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as xlw:
        df.to_excel(xlw, sheet_name="Inventory", index=False)

        # Optional: light formatting (freeze header, set widths)
        ws = xlw.sheets["Inventory"]
        ws.freeze_panes = "A2"  # freeze header row

        widths = {
            "A": 18,  # Received By
            "B": 12,  # B/L No
            "C": 12,  # B/L Date
            "D": 36,  # Product
            "E": 22,  # Packaging
            "F": 10,  # Quantity
            "G": 20,  # Lot No
            "H": 20,  # Ship To City/State
            "I": 30,  # Notes
        }
        for col, width in widths.items():
            ws.column_dimensions[col].width = width

    buffer.seek(0)
    return buffer.getvalue()


# ---------------------------
# UI: file upload + processing
# ---------------------------
uploaded = st.file_uploader(
    "Drop PDF files here (you can select multiple)",
    type=["pdf"],
    accept_multiple_files=True
)

if uploaded:
    st.info(f"{len(uploaded)} file(s) ready. Click **Process** to generate Excel.")

process_btn = st.button("Process", type="primary", disabled=not uploaded)

if process_btn and uploaded:
    df = process_bol_files(uploaded)
    st.success(f"Parsed {len(df)} line(s).")
    st.dataframe(df, use_container_width=True)

    excel_bytes = to_excel_bytes(df)
    st.download_button(
        label="⬇️ Download Excel",
        data=excel_bytes,
        file_name="IB_Inventory_AutoExtract.xlsx",
        mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )

st.caption("Tip: In Outlook, drag the PDF attachments to your desktop, then drop them here.")

<class 'ModuleNotFoundError'>: No module named 'pandas'